# Ionospheric Data Exploration â€” Hyderabad
Connects to the Madrigal API and pulls available experiments and parameters for the Hyderabad region (lat 17.38Â°N, lon 78.48Â°E).

In [1]:
import datetime
import madrigalWeb.madrigalWeb as mad

# â”€â”€ Site config â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
MADRIGAL_URL = "http://cedar.openmadrigal.org"

# Hyderabad coordinates
LAT = 17.38
LON = 78.48

# User info required by Madrigal for non-anonymous downloads
USER_NAME  = "Prithvi Raghu"
USER_EMAIL = "prithviraghu080706@gmail.com"
USER_AFF   = "Vellore Institute of Technology"

print(f"Connecting to {MADRIGAL_URL} ...")
madDB = mad.MadrigalData(MADRIGAL_URL)
print("Connected.")

Connecting to http://cedar.openmadrigal.org ...
Connected.


## 1  List all available instruments

In [2]:
instruments = madDB.getAllInstruments()

print(f"{'Code':<8} {'Category':<30} {'Name'}")
print("-" * 80)
for inst in instruments:
    print(f"{inst.code:<8} {inst.category:<30} {inst.name}")

Code     Category                       Name
--------------------------------------------------------------------------------
10       Incoherent Scatter Radars      Jicamarca IS Radar
11       Coherent Scatter Radars        Jicamarca Bistatic Radar
12       Incoherent Scatter Radars      JULIA MP ISR
13       Meteor Radars                  JASMET Jicamarca All-Sky Specular Meteor Radar
14       Coherent Scatter Radars        JULIA MP CSR
20       Incoherent Scatter Radars      Arecibo IS Radar - Linefeed
21       Incoherent Scatter Radars      Arecibo IS Radar - Gregorian
22       Incoherent Scatter Radars      Arecibo IS Radar - Velocity Vector
25       Incoherent Scatter Radars      MU IS Radar
30       Incoherent Scatter Radars      Millstone Hill IS Radar
31       Incoherent Scatter Radars      Millstone Hill UHF Steerable Antenna
32       Incoherent Scatter Radars      Millstone Hill UHF Zenith Antenna
40       Incoherent Scatter Radars      St. Santin IS Radar
41       Incoheren

## 2  Find instruments near Hyderabad
Filter the instrument list by geographic proximity (within Â±10Â° lat/lon).

In [3]:
LAT_TOL = 10.0
LON_TOL = 10.0

nearby = [
    inst for inst in instruments
    if abs(inst.latitude  - LAT) <= LAT_TOL
    and abs(inst.longitude - LON) <= LON_TOL
]

if nearby:
    print(f"Instruments within Â±{LAT_TOL}Â° of Hyderabad:")
    for inst in nearby:
        print(f"  [{inst.code}] {inst.name}  "
              f"(lat={inst.latitude:.2f}, lon={inst.longitude:.2f})")
else:
    print("No instruments found within tolerance â€” widening to global list.")
    nearby = instruments

nearby_codes = [inst.code for inst in nearby]

Instruments within Â±10.0Â° of Hyderabad:
  [1254] Tirunelveli MF radar  (lat=8.67, lon=77.82)
  [8319] Calcutta SCINDA Scintillation Receiver  (lat=22.58, lon=88.37)
  [8349] Rajkot SCINDA Scintillation Receiver  (lat=22.30, lon=70.80)
  [8356] Tirunelveli SCINDA Scintillation Receiver  (lat=8.67, lon=77.81)


## 3  Query experiments for nearby instruments

In [4]:
START = datetime.datetime(2015, 1, 1)
END   = datetime.datetime(2023, 1, 1)

print(f"Checking all nearby instruments for data between {START.date()} and {END.date()}...\n")

for code in nearby_codes:
    experiments = madDB.getExperiments(
        code,
        START.year, START.month, START.day,
        START.hour, START.minute, START.second,
        END.year, END.month, END.day,
        END.hour, END.minute, END.second,
        local=0
    )
    print(f"Code {code}: Found {len(experiments)} experiment(s)")
    if len(experiments) > 0:
        exp = experiments[0]
        print(f"  Available attributes: {[a for a in dir(exp) if not a.startswith('_')]}")
    print()

Checking all nearby instruments for data between 2015-01-01 and 2023-01-01...

Code 1254: Found 0 experiment(s)

Code 8319: Found 2437 experiment(s)
  Available attributes: ['access', 'endday', 'endhour', 'endmin', 'endmonth', 'endsec', 'endyear', 'id', 'instcode', 'instname', 'isLocal', 'madrigalUrl', 'name', 'pi', 'piEmail', 'realUrl', 'siteid', 'sitename', 'startday', 'starthour', 'startmin', 'startmonth', 'startsec', 'startyear', 'url', 'uttimestamp', 'version']

Code 8349: Found 346 experiment(s)
  Available attributes: ['access', 'endday', 'endhour', 'endmin', 'endmonth', 'endsec', 'endyear', 'id', 'instcode', 'instname', 'isLocal', 'madrigalUrl', 'name', 'pi', 'piEmail', 'realUrl', 'siteid', 'sitename', 'startday', 'starthour', 'startmin', 'startmonth', 'startsec', 'startyear', 'url', 'uttimestamp', 'version']

Code 8356: Found 1359 experiment(s)
  Available attributes: ['access', 'endday', 'endhour', 'endmin', 'endmonth', 'endsec', 'endyear', 'id', 'instcode', 'instname', 'isLo

## 3 Query experiments for Calcutta SCINDA (Code 8319)


In [5]:
# Use Calcutta SCINDA - most data available
INSTRUMENT_CODE = 8319

experiments = madDB.getExperiments(
    INSTRUMENT_CODE,
    2015, 1, 1, 0, 0, 0,
    2023, 1, 1, 0, 0, 0,
    local=0
)

print(f"Total experiments: {len(experiments)}\n")
for exp in experiments[:10]:
    print(f"ID: {exp.id} | {exp.startyear}-{exp.startmonth:02d}-{exp.startday:02d} to {exp.endyear}-{exp.endmonth:02d}-{exp.endday:02d}")

Total experiments: 2437

ID: 100282524 | 2015-01-01 to 2015-01-01
ID: 100282494 | 2015-01-02 to 2015-01-02
ID: 100282595 | 2015-01-03 to 2015-01-03
ID: 100282523 | 2015-01-04 to 2015-01-04
ID: 100282594 | 2015-01-05 to 2015-01-05
ID: 100282508 | 2015-01-06 to 2015-01-06
ID: 100282638 | 2015-01-07 to 2015-01-07
ID: 100282507 | 2015-01-08 to 2015-01-08
ID: 100282575 | 2015-01-09 to 2015-01-09
ID: 100282659 | 2015-01-10 to 2015-01-10


## 4  Inspect files and parameters for the first experiment

In [6]:
exp = experiments[0]

print(f"Experiment : {exp.name}")
print(f"Instrument : {exp.instname}")
print(f"Period     : {exp.startyear}-{exp.startmonth:02d}-{exp.startday:02d} â†’ {exp.endyear}-{exp.endmonth:02d}-{exp.endday:02d}")
print(f"ID         : {exp.id}\n")

files = madDB.getExperimentFiles(exp.id)
print(f"Files ({len(files)}):")
for f in files:
    print(f"  {f.name}  [{f.category}]")

if files:
    # print all available methods to find the right one
    methods = [m for m in dir(madDB) if 'aram' in m.lower() or 'param' in m.lower()]
    print(f"\nAvailable parameter methods: {methods}")

Experiment : SCINDA scintillation data
Instrument : Calcutta SCINDA Scintillation Receiver
Period     : 2015-01-01 â†’ 2015-01-01
ID         : 100282524

Files (1):
  /opt/openmadrigal/madroot/experiments4/2015/cal/01jan15/cal_gps_2015-01-01.01.hdf5  [1]

Available parameter methods: ['getExperimentFileParameters']


## 5 Ionospheric Measurements


In [7]:
params = madDB.getExperimentFileParameters(files[0].name)

print(f"Parameters in this file:\n")
print(f"{'Mnemonic':<20} {'Description'}")
print("-" * 70)
for p in params:
    print(f"{p.mnemonic:<20} {p.description}")

Parameters in this file:

Mnemonic             Description
----------------------------------------------------------------------
YEAR                 Year (universal time)
MONTH                Month (universal time)
DAY                  Day (universal time)
HOUR                 Hour (universal time)
MIN                  Minute (universal time)
SEC                  Second (universal time)
RECNO                Logical Record Number
KINDAT               Kind of data
KINST                Instrument Code
UT1_UNIX             Unix seconds (1/1/1970) at start
UT2_UNIX             Unix seconds (1/1/1970) at end
ANT_TYPE             Antenna type
RCVR_TYPE            Receiver type
PRN                  Pseudorandom noise code
LINK_FREQ            Frequency of satellite link signal
GNSS_TYPE            GNSS sat type (8 char padded str GPS or GLONASS)
PIERCE_ALT           Pierce Point Altitude
GDLATR               Reference geod latitude (N hemi=pos)
GDLONR               Reference geographic longi

## 5  Summary
This notebook identified Madrigal instruments near Hyderabad, listed available experiments over the chosen date range, and printed the parameters available in the first experiment file. Use the experiment IDs and parameter mnemonics in subsequent notebooks to download and model the data.

## 6. Load Data into Pandas DataFrame
Read key ionospheric and space weather parameters from the first experiment file

In [8]:
import pandas as pd

file_name = files[0].name

parms = 'year,month,day,hour,slt,szen,s4_scin,kp,dst,f10.7,swden,swspd,bimf'

data = madDB.isprint(
    file_name,
    parms,
    '',
    'Prithvi Raghu',
    'prithviraghu080706@gmail.com',
    'Vellore Institute of Technology'
)

print(data[:2000])

     2015         1         1         0          nan        nan           nan       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0      5.81501     102.07   3.00000e-02       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0          nan        nan           nan       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0          nan        nan           nan       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0      5.86321     100.02   5.00000e-02       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0      5.97348     100.55   6.00000e-02       2.30     -14.00   1.37300e+02  3200000.0   566000.0   5.96055e-09  
     2015         1         1         0          nan        nan           nan       2.30

## 7. Load Data into Pandas DataFrame
Convert the raw isprint output into a structured DataFrame, drop rows with missing S4 scintillation values, and preview the dataset with basic statistics.

In [9]:
from io import StringIO

# Convert isprint output to a proper dataframe
columns = ['year', 'month', 'day', 'hour', 'slt', 'szen', 's4_scin', 'kp', 'dst', 'f10.7', 'swden', 'swspd', 'bimf']

df = pd.read_csv(StringIO(data), sep='\s+', names=columns, na_values=['nan', 'missing'])

# Drop rows where s4_scin is missing since that's our key parameter
df = df.dropna(subset=['s4_scin'])

print(f"Shape: {df.shape}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nBasic stats:")
print(df.describe())

Shape: (14305, 13)

First 5 rows:


<>:6: SyntaxWarning: invalid escape sequence '\s'
<>:6: SyntaxWarning: invalid escape sequence '\s'
C:\Users\prith\AppData\Local\Temp\ipykernel_30428\2082914930.py:6: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(StringIO(data), sep='\s+', names=columns, na_values=['nan', 'missing'])


    year  month  day  hour      slt    szen  s4_scin   kp   dst  f10.7  \
1   2015      1    1     0  5.81501  102.07     0.03  2.3 -14.0  137.3   
4   2015      1    1     0  5.86321  100.02     0.05  2.3 -14.0  137.3   
5   2015      1    1     0  5.97348  100.55     0.06  2.3 -14.0  137.3   
8   2015      1    1     0  5.99734   99.35     0.05  2.3 -14.0  137.3   
10  2015      1    1     0  5.75888  103.21     0.04  2.3 -14.0  137.3   

        swden     swspd          bimf  
1   3200000.0  566000.0  5.960550e-09  
4   3200000.0  566000.0  5.960550e-09  
5   3200000.0  566000.0  5.960550e-09  
8   3200000.0  566000.0  5.960550e-09  
10  3200000.0  566000.0  5.960550e-09  

Basic stats:
          year    month      day          hour           slt          szen  \
count  14305.0  14305.0  14305.0  14305.000000  14305.000000  14305.000000   
mean    2015.0      1.0      1.0     11.737994     12.010944    104.354449   
std        0.0      0.0      0.0      6.873182      7.082259     41

## 8. Save Raw Data
Save the cleaned DataFrame to the data/raw directory as a CSV for future use.

In [10]:
import os
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/calcutta_scinda_2015_01_01.csv', index=False)
print(f"Saved {len(df)} rows to data/raw/calcutta_scinda_2015_01_01.csv")
print("Day 1 complete!")

Saved 14305 rows to data/raw/calcutta_scinda_2015_01_01.csv
Day 1 complete!


## 9. Build Full Dataset
Loop through all 2437 experiments from Calcutta SCINDA (2015-2023), pull key ionospheric and space weather parameters, and save as a single CSV.

In [11]:
import os
import time
from io import StringIO

os.makedirs('../data/raw', exist_ok=True)

parms = 'year,month,day,hour,slt,szen,s4_scin,kp,dst,f10.7,swden,swspd,bimf'
columns = ['year', 'month', 'day', 'hour', 'slt', 'szen', 's4_scin', 'kp', 'dst', 'f10.7', 'swden', 'swspd', 'bimf']

all_dfs = []
failed = []

# Sample every 15th experiment - gives ~163 days of data
sampled = experiments[::15]
print(f"Sampling {len(sampled)} experiments out of {len(experiments)}")

for i, exp in enumerate(sampled):
    try:
        files = madDB.getExperimentFiles(exp.id)
        if not files:
            continue
        
        raw = madDB.isprint(
            files[0].name,
            parms,
            '',
            'Prithvi Raghu',
            'prithviraghu080706@gmail.com',
            'Vellore Institute of Technology'
        )
        
        df_temp = pd.read_csv(StringIO(raw), sep=r'\s+', names=columns, na_values=['nan', 'missing'])
        df_temp = df_temp.dropna(subset=['s4_scin'])
        
        if len(df_temp) > 0:
            all_dfs.append(df_temp)

        # Save checkpoint every 10 experiments
        if i % 10 == 0 and len(all_dfs) > 0:
            pd.concat(all_dfs, ignore_index=True).to_csv('../data/raw/checkpoint.csv', index=False)
            
        if i % 25 == 0:
            print(f"Progress: {i}/{len(sampled)} | Rows so far: {sum(len(d) for d in all_dfs)}")
            
    except Exception as e:
        failed.append((exp.id, str(e)))
        continue

df_full = pd.concat(all_dfs, ignore_index=True)
df_full.to_csv('../data/raw/calcutta_scinda_sampled.csv', index=False)
print(f"\nDone! Total rows: {len(df_full)}")
print(f"Failed: {len(failed)}")
print(df_full.head())

Sampling 163 experiments out of 2437
Progress: 0/163 | Rows so far: 14305
Progress: 25/163 | Rows so far: 364177
Progress: 50/163 | Rows so far: 720659
Progress: 75/163 | Rows so far: 1049860
Progress: 100/163 | Rows so far: 1353618
Progress: 125/163 | Rows so far: 1615004
Progress: 150/163 | Rows so far: 1844267

Done! Total rows: 1958270
Failed: 0
   year  month  day  hour      slt    szen  s4_scin   kp   dst  f10.7  \
0  2015      1    1     0  5.81501  102.07     0.03  2.3 -14.0  137.3   
1  2015      1    1     0  5.86321  100.02     0.05  2.3 -14.0  137.3   
2  2015      1    1     0  5.97348  100.55     0.06  2.3 -14.0  137.3   
3  2015      1    1     0  5.99734   99.35     0.05  2.3 -14.0  137.3   
4  2015      1    1     0  5.75888  103.21     0.04  2.3 -14.0  137.3   

       swden     swspd          bimf  
0  3200000.0  566000.0  5.960550e-09  
1  3200000.0  566000.0  5.960550e-09  
2  3200000.0  566000.0  5.960550e-09  
3  3200000.0  566000.0  5.960550e-09  
4  3200000.0  

## 10. Preprocessing & Save to Processed
Create datetime column, drop remaining NaNs, fix bimf units, normalize key columns, and save to data/processed/

In [12]:
import pandas as pd
import numpy as np
import os

# Load raw data
df = pd.read_csv('../data/raw/calcutta_scinda_sampled.csv')

print('Shape before cleaning:', df.shape)
print('\nMissing values per column:')
print(df.isnull().sum())

# 1. Datetime column
df['datetime'] = pd.to_datetime(dict(
    year=df['year'], month=df['month'], day=df['day'], hour=df['hour']
))

# 2. Drop remaining NaNs
df_clean = df.dropna().copy()
print(f'\nShape after dropping remaining NaNs: {df_clean.shape}')

# 3. Sanity check suspicious columns
print('\nUnique values in swden:', df_clean['swden'].nunique())
print('Unique values in swspd:', df_clean['swspd'].nunique())
print('Unique values in bimf :', df_clean['bimf'].nunique())

# 4. Unit fix for bimf
if df_clean['bimf'].abs().max() < 1e-6:
    df_clean['bimf'] = df_clean['bimf'] * 1e9
    print('Converted bimf from Tesla to nanotesla')

# 5. Normalize numerical columns
cols_to_normalize = ['slt', 'szen', 's4_scin', 'kp', 'dst', 'f10.7', 'swden', 'swspd', 'bimf']

for col in cols_to_normalize:
    min_val = df_clean[col].min()
    max_val = df_clean[col].max()
    if max_val > min_val:
        df_clean[f'{col}_norm'] = (df_clean[col] - min_val) / (max_val - min_val)
    else:
        print(f'WARNING: {col} has no variance â€” skipping normalization')

# 6. Save
os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/calcutta_scinda_processed.csv', index=False)
print(f'\nSaved {len(df_clean)} rows to data/processed/calcutta_scinda_processed.csv')
print(df_clean[['datetime', 's4_scin', 'kp', 'dst', 'bimf']].head())

Shape before cleaning: (1958270, 13)

Missing values per column:
year          0
month         0
day           0
hour          0
slt           0
szen          0
s4_scin       0
kp            0
dst           0
f10.7         0
swden      2581
swspd       531
bimf          0
dtype: int64

Shape after dropping remaining NaNs: (1955689, 14)

Unique values in swden: 240
Unique values in swspd: 419
Unique values in bimf : 2460
Converted bimf from Tesla to nanotesla

Saved 1955689 rows to data/processed/calcutta_scinda_processed.csv
    datetime  s4_scin   kp   dst     bimf
0 2015-01-01     0.03  2.3 -14.0  5.96055
1 2015-01-01     0.05  2.3 -14.0  5.96055
2 2015-01-01     0.06  2.3 -14.0  5.96055
3 2015-01-01     0.05  2.3 -14.0  5.96055
4 2015-01-01     0.04  2.3 -14.0  5.96055


## 11. Hybrid Model Selector Test

In [1]:
import sys
sys.path.insert(0, '..')

from models.hybrid_selector import select_model

tests = [
    (22.5, 6.0,  -50.0, False),
    (22.5, 1.0, -200.0, False),
    (70.0, 1.0,  -20.0, False),
    (22.5, 1.0,  -20.0, True),
    (22.5, 1.0,  -20.0, False),
]

for lat, kp, dst, irtam in tests:
    r = select_model(lat, kp, dst, irtam)
    print(f"lat={lat:>5}, Kp={kp}, Dst={dst:>6}, IRTAM={irtam} â†’ {r.model:8} | {r.reason}")

lat= 22.5, Kp=6.0, Dst= -50.0, IRTAM=False â†’ SAMI3    | Storm conditions: Kp=6.0, Dst=-50.0
lat= 22.5, Kp=1.0, Dst=-200.0, IRTAM=False â†’ SAMI3    | Storm conditions: Kp=1.0, Dst=-200.0
lat= 70.0, Kp=1.0, Dst= -20.0, IRTAM=False â†’ E-CHAIM  | High latitude: lat=70.0Â°
lat= 22.5, Kp=1.0, Dst= -20.0, IRTAM=True â†’ IRTAM    | IRTAM available and conditions are nominal
lat= 22.5, Kp=1.0, Dst= -20.0, IRTAM=False â†’ IRI      | Calm conditions (Kp=1.0, Dst=-20.0), IRTAM unavailable


## 12. Hybrid Model Integration Test

In [2]:
import sys
sys.path.insert(0, '..')

from datetime import datetime
from models.hybrid_model import get_ionosphere

# Calcutta coordinates
lat, lon = 22.5, 88.3
dt = datetime(2020, 6, 15, 12, 0, 0)

test_cases = [
    {"kp": 1.0, "dst": -20.0,  "irtam": False, "label": "Calm → IRI"},
    {"kp": 6.0, "dst": -50.0,  "irtam": False, "label": "Storm → SAMI3 (IRI fallback)"},
    {"kp": 1.0, "dst": -200.0, "irtam": False, "label": "Dst storm → SAMI3 (IRI fallback)"},
    {"kp": 1.0, "dst": -20.0,  "irtam": True,  "label": "IRTAM available (IRI fallback)"},
]

for tc in test_cases:
    result = get_ionosphere(lat, lon, dt, tc["kp"], tc["dst"], tc["irtam"])
    p = result["profile"]
    print(f"[{tc['label']}]")
    print(f"  Model used : {result['model_used']}")
    print(f"  Reason     : {result['reason']}")
    print(f"  NmF2       : {p.NmF2:.3e} m⁻³")
    print(f"  hmF2       : {p.hmF2:.1f} km")
    print(f"  foF2       : {p.foF2:.1f} MHz")
    print(f"  TEC        : {p.TEC:.2f} TECU")


c:\Users\prith\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\prith\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
c:\Users\prith\anaconda3\Lib\site-packages\iri2016\base.py:63: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  iono = xarray

[Calm → IRI]
  Model used : IRI
  Reason     : Calm conditions (Kp=1.0, Dst=-20.0), IRTAM unavailable
  NmF2       : 1.369e+12 m⁻³
  hmF2       : 280.0 km
  foF2       : 10507.4 MHz
  TEC        : 0.03 TECU
[Storm → SAMI3 (IRI fallback)]
  Model used : SAMI3
  Reason     : Storm conditions: Kp=6.0, Dst=-50.0
  NmF2       : 1.369e+12 m⁻³
  hmF2       : 280.0 km
  foF2       : 10507.4 MHz
  TEC        : 0.03 TECU


c:\Users\prith\anaconda3\Lib\site-packages\iri2016\base.py:63: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  iono = xarray.Dataset(
IRTAM selected but not yet implemented — falling back to IRI


[Dst storm → SAMI3 (IRI fallback)]
  Model used : SAMI3
  Reason     : Storm conditions: Kp=1.0, Dst=-200.0
  NmF2       : 1.369e+12 m⁻³
  hmF2       : 280.0 km
  foF2       : 10507.4 MHz
  TEC        : 0.03 TECU
[IRTAM available (IRI fallback)]
  Model used : IRTAM
  Reason     : IRTAM available and conditions are nominal
  NmF2       : 1.369e+12 m⁻³
  hmF2       : 280.0 km
  foF2       : 10507.4 MHz
  TEC        : 0.03 TECU


c:\Users\prith\anaconda3\Lib\site-packages\iri2016\base.py:63: UserWarning: Converting non-nanosecond precision datetime values to nanosecond precision. This behavior can eventually be relaxed in xarray, as it is an artifact from pandas which is now beginning to support non-nanosecond precision values. This warning is caused by passing non-nanosecond np.datetime64 or np.timedelta64 values to the DataArray or Variable constructor; it can be silenced by converting the values to nanosecond precision ahead of time.
  iono = xarray.Dataset(


## 13. SSL Algorithm Test

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import datetime
from models.ssl_algorithm import ssl_locate

# Calcutta receiver
RECEIVER_LAT = 22.5
RECEIVER_LON  = 88.3

test_cases = [
    {"az": 45,  "el": 15, "freq": 10.0, "kp": 1.0, "dst": -20.0,  "label": "NE, calm"},
    {"az": 270, "el": 20, "freq": 15.0, "kp": 1.0, "dst": -20.0,  "label": "W, calm"},
    {"az": 0,   "el": 10, "freq": 8.0,  "kp": 6.0, "dst": -50.0,  "label": "N, storm"},
    {"az": 135, "el": 30, "freq": 20.0, "kp": 1.0, "dst": -200.0, "label": "SE, Dst storm"},
]

dt = datetime(2020, 6, 15, 12, 0, 0)

for tc in test_cases:
    r = ssl_locate(
        receiver_lat=RECEIVER_LAT,
        receiver_lon=RECEIVER_LON,
        azimuth_deg=tc["az"],
        elevation_deg=tc["el"],
        frequency_mhz=tc["freq"],
        dt=dt,
        kp=tc["kp"],
        dst=tc["dst"],
        irtam_available=False
    )
    print(f"
[{tc['label']}]")
    print(f"  Azimuth / Elevation : {tc['az']}° / {tc['el']}°")
    print(f"  Virtual Height      : {r.virtual_height_km} km")
    print(f"  Ground Distance     : {r.ground_distance_km} km")
    print(f"  Transmitter Location: ({r.transmitter_lat}, {r.transmitter_lon})")
    print(f"  Model Used          : {r.model_used}")
